# Financial news ticker linking benchmark (Kaggle)

Standalone multi-label text-classification notebook. Attach the benchmark CSV dataset as a Kaggle **Input**; the notebook finds `financial_news_ticker_linking.csv` under `/kaggle/input`.

Each article is paired with zero or more ticker targets from `mentioned_companies`. The model predicts a score for every ticker in the fixed 49-ticker vocabulary and compares FinBERT, FLANG-RoBERTa, and FLANG-DistilBERT.

The CSV contains 49,128 training rows. Defaults cap training at 10,000 rows and validation/test at 3,000 rows each for an initial Kaggle run. Tokenization is batched once per model and training/evaluation show progress bars. Set `TRAIN_ROW_LIMIT = None` and `EVAL_ROW_LIMIT = None` only after the capped run works; full multi-model training is substantially longer. Internet must be enabled for first-time model downloads.


## Dependencies

Run the next cell first. Kaggle usually provides CUDA-enabled PyTorch; we leave that version installed. If package installation changes a package already imported by the session, restart and run the notebook from the top.

In [ ]:
%pip install -q "transformers<5" sentencepiece numpy pandas scikit-learn matplotlib tqdm

In [ ]:
from pathlib import Path
import ast
import json
import random
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (average_precision_score, f1_score, hamming_loss,
                             precision_score, recall_score)
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
INPUT_ROOT = Path("/kaggle/input")
matches = list(INPUT_ROOT.rglob("financial_news_ticker_linking.csv")) if INPUT_ROOT.exists() else []
if not matches:
    raise FileNotFoundError("Could not find financial_news_ticker_linking.csv under /kaggle/input. Attach the benchmark dataset as a Kaggle Input.")
DATA_PATH = matches[0]
OUTPUT_DIR = Path("/kaggle/working/financial_ticker_linking")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CHECKPOINTS = {
    "FinBERT": "yiyanghkust/finbert-pretrain",
    "FLANG-RoBERTa": "SALT-NLP/FLANG-Roberta",
    "FLANG-DistilBERT": "SALT-NLP/FLANG-DistilBERT",
}
SEED = 42
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
MAX_LENGTH = 192
BATCH_SIZE = 8  # Lower to 4 if your GPU runs out of memory.
EPOCHS = 3
ENCODER_LR = 2e-5
HEAD_LR = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
DROPOUT = 0.25
HEAD_HIDDEN_SIZES = (256,)
PATIENCE = 1
MAX_TEXT_CHARS = 2000  # Longer text is truncated to 192 tokens anyway.
TRAIN_ROW_LIMIT = 10000  # Set to None for all 49k+ train rows after a capped run works.
EVAL_ROW_LIMIT = 3000   # Set to None to evaluate full validation/test partitions.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
SAVE_CHECKPOINTS = False
print("Data:", DATA_PATH)
print("Device:", DEVICE)
print("Model checkpoints:", MODEL_CHECKPOINTS)
print("Kaggle Internet must be enabled for first-time model downloads.")

In [ ]:
import torch
from torch import nn
from transformers import AutoModel

class MultiLabelClassificationHead(nn.Module):
    """MLP prediction head; outputs one logit per ticker label."""
    def __init__(self, hidden_size, num_labels, dropout=0.25, hidden_sizes=(256,)):
        super().__init__()
        layers, input_size = [], hidden_size
        for width in hidden_sizes:
            layers.extend([nn.Linear(input_size, width), nn.LayerNorm(width), nn.GELU(), nn.Dropout(dropout)])
            input_size = width
        layers.append(nn.Linear(input_size, num_labels))
        self.network = nn.Sequential(*layers)
    def forward(self, pooled):
        return self.network(pooled)

class FinancialTransformerBase(nn.Module):
    """Shared pretrained encoder composition for financial text tasks."""
    def __init__(self, checkpoint, task_head):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(checkpoint)
        config = self.encoder.config
        hidden_size = getattr(config, "hidden_size", None) or getattr(config, "dim", None) or getattr(config, "d_model", None)
        if hidden_size is None:
            raise ValueError(f"Could not infer hidden size for {checkpoint}")
        self.head = task_head(int(hidden_size))
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        inputs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            inputs["token_type_ids"] = token_type_ids
        output = self.encoder(**inputs)
        pooled = getattr(output, "pooler_output", None)
        if pooled is None:
            pooled = output.last_hidden_state[:, 0]
        return self.head(pooled)

class FinancialTextMultiLabelClassifier(FinancialTransformerBase):
    def __init__(self, checkpoint, num_labels, dropout=DROPOUT, hidden_sizes=HEAD_HIDDEN_SIZES):
        super().__init__(checkpoint, lambda size: MultiLabelClassificationHead(size, num_labels, dropout, hidden_sizes))

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def parse_tickers(value):
    if pd.isna(value) or not str(value).strip():
        return []
    try:
        items = json.loads(value)
    except (json.JSONDecodeError, TypeError):
        try:
            items = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    if isinstance(items, str):
        items = [items]
    if not isinstance(items, (list, tuple, set)):
        return []
    return sorted({str(item).strip().upper().lstrip("$") for item in items if str(item).strip()})


def normalize_title(value):
    value = str(value or "").lower()
    return re.sub(r"[^a-z0-9]+", " ", value).strip()


def prepare_dataframe():
    required = ["text", "title", "description", "date_publish", "url", "mentioned_companies"]
    frame = pd.read_csv(DATA_PATH, usecols=required, low_memory=False, encoding="utf-8-sig")
    frame["tickers"] = frame["mentioned_companies"].apply(parse_tickers)
    frame = frame[frame["tickers"].map(len) > 0].copy()
    frame["date"] = pd.to_datetime(frame["date_publish"], errors="coerce", utc=True)
    frame["url"] = frame["url"].fillna("").astype(str).str.strip()
    frame["title"] = frame["title"].fillna("").astype(str)
    frame["description"] = frame["description"].fillna("").astype(str)
    frame["text"] = frame["text"].fillna("").astype(str)
    frame = frame.sort_values("date", kind="stable", na_position="first")
    # Drop repeated URLs; keep the earliest occurrence in this local snapshot.
    has_url = frame["url"].ne("")
    unique_url_rows = frame.loc[has_url].drop_duplicates("url", keep="first")
    no_url_rows = frame.loc[~has_url]
    frame = pd.concat([unique_url_rows, no_url_rows]).sort_values("date", kind="stable", na_position="first")
    frame["title_key"] = frame["title"].map(normalize_title)
    # Group identical headlines so a repeated article title cannot cross temporal partitions.
    row_ids = np.arange(len(frame))
    frame["group_key"] = [f"title:{key}" if key else f"row:{i}" for key, i in zip(frame["title_key"], row_ids)]
    frame["model_text"] = (
        "Title: " + frame["title"] + "\nSummary: " + frame["description"] + "\nArticle: " + frame["text"]
    ).str.slice(0, MAX_TEXT_CHARS)
    return frame.reset_index(drop=True)


def make_temporal_splits(frame):
    # Use each duplicate-headline group's latest date to place the whole group in one split.
    group_dates = frame.groupby("group_key", sort=False)["date"].max().dropna().sort_values(kind="stable")
    if len(group_dates) < 3:
        raise ValueError("Too few dated headline groups for temporal train/validation/test splitting.")
    train_cut = group_dates.iloc[min(int(len(group_dates) * TRAIN_FRACTION), len(group_dates) - 2)]
    val_cut_index = min(int(len(group_dates) * (TRAIN_FRACTION + VALIDATION_FRACTION)), len(group_dates) - 1)
    val_cut = group_dates.iloc[val_cut_index]
    max_date_by_group = frame.groupby("group_key", sort=False)["date"].max()
    group_split = {}
    for group, group_date in max_date_by_group.items():
        if pd.isna(group_date) or group_date <= train_cut:
            group_split[group] = "train"
        elif group_date <= val_cut:
            group_split[group] = "validation"
        else:
            group_split[group] = "test"
    frame["split"] = frame["group_key"].map(group_split)
    # Rows with unknown dates are training-only; they are not used for temporal evaluation.
    return {name: frame[frame["split"] == name].reset_index(drop=True)
            for name in ["train", "validation", "test"]}


frame = prepare_dataframe()
label_names = sorted({ticker for tickers in frame["tickers"] for ticker in tickers})
label_to_id = {name: i for i, name in enumerate(label_names)}
for ticker in label_names:
    pass
frame["target_vector"] = frame["tickers"].apply(
    lambda tickers: np.array([1.0 if ticker in tickers else 0.0 for ticker in label_names], dtype=np.float32)
)
splits = make_temporal_splits(frame)

# Optional deterministic caps for quick experiments. Leave None for full training/evaluation.
def cap_frame(part, limit, seed=SEED):
    if limit is not None and len(part) > limit:
        return part.sample(n=limit, random_state=seed).sort_index().reset_index(drop=True)
    return part.reset_index(drop=True)

splits["train"] = cap_frame(splits["train"], TRAIN_ROW_LIMIT)
splits["validation"] = cap_frame(splits["validation"], EVAL_ROW_LIMIT)
splits["test"] = cap_frame(splits["test"], EVAL_ROW_LIMIT)
if any(part.empty for part in splits.values()):
    raise ValueError("A temporal split is empty; inspect publication dates and split fractions.")

print("Rows by split:", {name: len(part) for name, part in splits.items()})
print("Distinct ticker labels:", len(label_names), label_names)
print("Publication-date ranges:", {name: (part["date"].min(), part["date"].max()) for name, part in splits.items()})
train_labels = {ticker for tickers in splits["train"]["tickers"] for ticker in tickers}
test_labels = {ticker for tickers in splits["test"]["tickers"] for ticker in tickers}
print("Ticker labels present in test but absent from train:", sorted(test_labels - train_labels))
print("Note: label vocabulary is fixed from this corpus; unseen-in-training tickers cannot be learned by the model.")

In [ ]:
class NewsTickerDataset(Dataset):
    """Tokenize each split once in batches, instead of re-tokenizing on every epoch."""
    def __init__(self, part, tokenizer, max_length, split_name="data"):
        texts=part["model_text"].astype(str).tolist()
        self.targets=np.stack(part["target_vector"].to_numpy()).astype(np.float32)
        self.examples=[]
        chunk_size=512
        for start in tqdm(range(0,len(texts),chunk_size),desc=f"Tokenizing {split_name}",leave=True):
            encoded=tokenizer(texts[start:start+chunk_size],truncation=True,max_length=max_length,padding=False)
            keys=list(encoded.keys())
            self.examples.extend([{key:encoded[key][i] for key in keys} for i in range(len(encoded["input_ids"]))])
    def __len__(self):
        return len(self.examples)
    def __getitem__(self,index):
        item=dict(self.examples[index])
        item["labels"]=self.targets[index]
        return item

def make_collator(tokenizer):
    def collate(items):
        labels = torch.tensor(np.stack([item.pop("labels") for item in items]), dtype=torch.float32)
        batch = tokenizer.pad(items, padding=True, return_tensors="pt")
        batch["labels"] = labels
        return batch
    return collate


def make_loader(part, tokenizer, max_length, shuffle=False, split_name="data"):
    dataset=NewsTickerDataset(part,tokenizer,max_length,split_name=split_name)
    return DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=shuffle,
                      num_workers=0,pin_memory=DEVICE.type=="cuda",collate_fn=make_collator(tokenizer))


def probabilities_and_targets(model, loader):
    model.eval()
    actual, probabilities = [], []
    with torch.no_grad():
        for batch in tqdm(loader,desc="Evaluating",leave=False):
            labels = batch.pop("labels")
            batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(**batch)
            actual.append(labels.numpy())
            probabilities.append(torch.sigmoid(logits).float().cpu().numpy())
    return np.concatenate(actual), np.concatenate(probabilities)


def multilabel_metrics(actual, predicted, probabilities=None):
    result = {
        "micro_f1": f1_score(actual, predicted, average="micro", zero_division=0),
        "macro_f1": f1_score(actual, predicted, average="macro", zero_division=0),
        "micro_precision": precision_score(actual, predicted, average="micro", zero_division=0),
        "micro_recall": recall_score(actual, predicted, average="micro", zero_division=0),
        "macro_precision": precision_score(actual, predicted, average="macro", zero_division=0),
        "macro_recall": recall_score(actual, predicted, average="macro", zero_division=0),
        "exact_match_accuracy": float(np.mean(np.all(actual == predicted, axis=1))),
        "hamming_loss": hamming_loss(actual, predicted),
    }
    if probabilities is not None:
        ap_values = [average_precision_score(actual[:, i], probabilities[:, i])
                     for i in range(actual.shape[1]) if actual[:, i].sum() > 0]
        result["macro_average_precision"] = float(np.mean(ap_values)) if ap_values else float("nan")
    return result


def tune_global_threshold(actual, probabilities):
    # Tune one threshold on validation only to avoid overfitting dozens of per-ticker thresholds.
    candidates = np.arange(0.10, 0.901, 0.025)
    scored = [(f1_score(actual, probabilities >= threshold, average="macro", zero_division=0), threshold)
              for threshold in candidates]
    return max(scored, key=lambda item: item[0])[1]

In [ ]:
def train_one_model(model_label, checkpoint):
    set_seed()
    print(f"Loading fast tokenizer for {model_label}: {checkpoint}",flush=True)
    try:
        tokenizer=AutoTokenizer.from_pretrained(checkpoint,use_fast=True)
    except Exception as exc:
        print(f"Fast tokenizer unavailable ({exc}); using slow tokenizer.",flush=True)
        tokenizer=AutoTokenizer.from_pretrained(checkpoint,use_fast=False)
    print(f"Loading encoder weights for {model_label}; first download may take a few minutes...",flush=True)
    model=FinancialTextMultiLabelClassifier(checkpoint,len(label_names)).to(DEVICE)
    print(f"{model_label} encoder loaded.",flush=True)
    # RoBERTa/BERTweet position IDs include a padding offset, so cap sequence
    # length to max_position_embeddings - pad_token_id - 1 (128 for FLANG-RoBERTa).
    config = model.encoder.config
    position_limit = getattr(config, "max_position_embeddings", MAX_LENGTH)
    if getattr(config, "model_type", "") in {"roberta", "bart"}:
        position_limit = position_limit - int(getattr(config, "pad_token_id", 1)) - 1
    tokenizer_limit = getattr(tokenizer, "model_max_length", MAX_LENGTH)
    if tokenizer_limit is None or tokenizer_limit > 100000:
        tokenizer_limit = MAX_LENGTH
    effective_max_length = min(MAX_LENGTH, int(position_limit), int(tokenizer_limit))
    if effective_max_length < 8:
        raise ValueError(f"Invalid maximum input length {effective_max_length} for {checkpoint}")
    embedding_rows = model.encoder.get_input_embeddings().num_embeddings
    largest_token_id = max(tokenizer.get_vocab().values())
    if largest_token_id >= embedding_rows:
        raise ValueError(f"Tokenizer vocab id {largest_token_id} exceeds {checkpoint} embedding rows ({embedding_rows}).")
    print(f"Using max_length={effective_max_length} (model positional limit={position_limit})",flush=True)
    print(f"Preparing tokenized rows: train={len(splits['train']):,}, validation={len(splits['validation']):,}, test={len(splits['test']):,}",flush=True)
    train_loader=make_loader(splits["train"],tokenizer,effective_max_length,shuffle=True,split_name="train")
    val_loader=make_loader(splits["validation"],tokenizer,effective_max_length,split_name="validation")
    test_loader=make_loader(splits["test"],tokenizer,effective_max_length,split_name="test")
    print(f"Tokenization complete. {len(train_loader):,} training batches per epoch at batch size {BATCH_SIZE}.",flush=True)

    target_matrix = np.stack(splits["train"]["target_vector"].to_numpy())
    positives = target_matrix.sum(axis=0)
    negatives = len(target_matrix) - positives
    pos_weight = np.sqrt(negatives / np.maximum(positives, 1.0))
    pos_weight = torch.tensor(np.clip(pos_weight, 1.0, 20.0), dtype=torch.float32, device=DEVICE)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    head_params = list(model.head.parameters())
    head_ids = {id(param) for param in head_params}
    encoder_params = [param for param in model.parameters() if id(param) not in head_ids]
    optimizer = torch.optim.AdamW([
        {"params": encoder_params, "lr": ENCODER_LR},
        {"params": head_params, "lr": HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)
    total_steps = max(1, len(train_loader) * EPOCHS)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    best_val_f1, best_state, stale, history = -1.0, None, 0, []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum, seen = 0.0, 0
        print(f"Starting epoch {epoch}/{EPOCHS} for {model_label}...",flush=True)
        for batch in tqdm(train_loader,desc=f"{model_label} epoch {epoch}/{EPOCHS}",leave=True):
            labels = batch.pop("labels").to(DEVICE, non_blocking=True)
            batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(**batch)
                loss = loss_fn(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            loss_sum += loss.item() * labels.size(0)
            seen += labels.size(0)

        y_val, p_val = probabilities_and_targets(model, val_loader)
        val_metrics = multilabel_metrics(y_val, p_val >= 0.5, p_val)
        row = {"epoch": epoch, "train_loss": loss_sum / max(seen, 1),
               **{f"val_{key}": value for key, value in val_metrics.items()}}
        history.append(row)
        print(f"{model_label} | epoch {epoch}: train_loss={row['train_loss']:.4f}, val_micro_f1={val_metrics['micro_f1']:.4f}, val_macro_f1={val_metrics['macro_f1']:.4f}")
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print("Early stopping: validation macro-F1 did not improve.")
                break

    model.load_state_dict(best_state)
    y_val, p_val = probabilities_and_targets(model, val_loader)
    threshold = tune_global_threshold(y_val, p_val)
    y_test, p_test = probabilities_and_targets(model, test_loader)
    y_pred = (p_test >= threshold).astype(np.int32)
    result = {"model": model_label, "checkpoint": checkpoint, "threshold": threshold,
              "train_rows": len(splits["train"]), "validation_rows": len(splits["validation"]),
              "test_rows": len(splits["test"]), **multilabel_metrics(y_test, y_pred, p_test)}
    per_label = []
    for i, ticker in enumerate(label_names):
        per_label.append({"model": model_label, "ticker": ticker,
                          "test_support": int(y_test[:, i].sum()),
                          "precision": precision_score(y_test[:, i], y_pred[:, i], zero_division=0),
                          "recall": recall_score(y_test[:, i], y_pred[:, i], zero_division=0),
                          "f1": f1_score(y_test[:, i], y_pred[:, i], zero_division=0)})
    if SAVE_CHECKPOINTS:
        torch.save(best_state, OUTPUT_DIR / f"ticker_linking_{model_label.lower().replace('-', '_')}.pt")
    del model, optimizer, scheduler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result, pd.DataFrame(history), (y_test, y_pred, p_test), pd.DataFrame(per_label)

results, histories, predictions, per_label_results = [], {}, {}, []
for model_label, checkpoint in MODEL_CHECKPOINTS.items():
    result, history, prediction_data, label_metrics = train_one_model(model_label, checkpoint)
    results.append(result)
    histories[model_label] = history
    predictions[model_label] = prediction_data
    per_label_results.append(label_metrics)
results_df = pd.DataFrame(results).sort_values("micro_f1", ascending=False).reset_index(drop=True)
per_label_df = pd.concat(per_label_results, ignore_index=True)
results_df

In [ ]:
results_df.to_csv(OUTPUT_DIR / "ticker_linking_metrics.csv", index=False)
per_label_df.to_csv(OUTPUT_DIR / "ticker_linking_per_ticker_metrics.csv", index=False)
print("Saved result tables to:", OUTPUT_DIR)

## Visualizations

In [ ]:
# Overall held-out multi-label metrics.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
results_df.set_index("model")[["micro_f1", "macro_f1"]].plot(kind="bar", ax=axes[0], rot=15)
axes[0].set(title="Held-out F1 scores", ylabel="F1", ylim=(0, 1))
results_df.set_index("model")[["micro_precision", "micro_recall"]].plot(kind="bar", ax=axes[1], rot=15)
axes[1].set(title="Held-out micro precision/recall", ylabel="Score", ylim=(0, 1))
for ax in axes:
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "ticker_linking_model_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

# Training loss and validation learning curves.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for model_label, history in histories.items():
    axes[0].plot(history["epoch"], history["train_loss"], marker="o", label=model_label)
    axes[1].plot(history["epoch"], history["val_macro_f1"], marker="o", label=model_label)
axes[0].set(title="Training BCE loss", xlabel="Epoch", ylabel="Loss")
axes[1].set(title="Validation macro-F1 at 0.5 threshold", xlabel="Epoch", ylabel="Macro-F1")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "ticker_linking_learning_curves.png", dpi=160, bbox_inches="tight")
plt.show()

# Per-ticker test F1 for the most common 15 tickers.
support = per_label_df.groupby("ticker")["test_support"].max().sort_values(ascending=False)
top_tickers = support.head(15).index
plot_data = per_label_df[per_label_df["ticker"].isin(top_tickers)].pivot(index="ticker", columns="model", values="f1").reindex(top_tickers)
fig, ax = plt.subplots(figsize=(12, 6))
plot_data.plot(kind="bar", ax=ax, rot=45)
ax.set(title="Per-ticker held-out F1 (top 15 by test support)", ylabel="F1", xlabel="Ticker")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "ticker_linking_per_ticker_f1.png", dpi=160, bbox_inches="tight")
plt.show()
print("Plots saved to:", OUTPUT_DIR)

## Metrics and interpretation

- **Micro-F1 / micro precision / micro recall** aggregate ticker decisions and are influenced most by common tickers.
- **Macro-F1 / macro precision / macro recall** average across tickers and give rare tickers equal weight.
- **Exact-match accuracy** requires every ticker in an article's predicted set to match exactly.
- **Hamming loss** measures the fraction of individual ticker decisions that are wrong.
- **Macro average precision** evaluates ranking quality across labels.

The global classification threshold is selected using validation macro-F1 only. Read `ticker_linking_per_ticker_metrics.csv` alongside aggregate scores because rare labels can have low support. The fixed ticker vocabulary is derived from this curated corpus; any label absent from training is necessarily difficult or impossible to learn from this split. Results are model comparisons for source annotations, not verification that every source ticker label is factually correct.